# UAS — Implementasi SVM, CNN, dan LSTM
## Studi Kasus: Klasifikasi Sentimen Ulasan Produk Tokopedia

**Dataset:** `tokopedia-product-reviews-2019.csv` (ulasan produk, kolom utama: `text` dan `rating`)

**Target:** Label sentimen diturunkan dari `rating`:
- Rating 1–2 → **Negative**
- Rating 3 → **Neutral**
- Rating 4–5 → **Positive**

Notebook ini berjalan di **Google Colab** dan membaca dataset serta modul pendukung langsung dari **Google Drive**.

## 1. Install & Import Library

In [ ]:
!pip install -q tensorflow scikit-learn nltk Sastrawi emoji wordcloud seaborn matplotlib pandas numpy

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)   # safeguard untuk versi NLTK terbaru
nltk.download("stopwords", quiet=True)

print("Library berhasil di-import.")

## 2. Mount Google Drive

Sebelum import modul, mount Google Drive terlebih dahulu, lalu arahkan ke folder project (berisi `eda.py`, `preprocessing.py`, `visualization.py`, folder `engine/`, dan file dataset `tokopedia-product-reviews-2019.csv`).

> **Catatan:** Upload folder `code/` (hasil ekstrak) ke Google Drive kamu terlebih dahulu, misal ke `MyDrive/UAS_SVM_CNN_LSTM/code/`. Sesuaikan `PROJECT_PATH` di bawah dengan lokasi folder tersebut di Drive kamu.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/code"  # sesuaikan dengan lokasi folder di Drive kamu

assert os.path.exists(PROJECT_PATH), f"Folder tidak ditemukan: {PROJECT_PATH}. Periksa kembali path-nya."

sys.path.append(PROJECT_PATH)
os.chdir(PROJECT_PATH)

print("Working directory:", os.getcwd())
print("Isi folder:", os.listdir(PROJECT_PATH))

## 3. Import Module

Mengimpor seluruh fungsi dari modul lokal (`preprocessing.py`, `eda.py`, `visualization.py`, dan `engine/`).

In [ ]:
from preprocessing import *
from eda import *
from visualization import *
from engine.svm_engine import *
from engine.cnn_engine import *
from engine.lstm_engine import *

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Semua modul berhasil di-import.")

## 4. Load Dataset

In [ ]:
DATA_PATH = os.path.join(PROJECT_PATH, "tokopedia-product-reviews-2019.csv")

df = pd.read_csv(DATA_PATH)

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print("Dataset berhasil dimuat.")

## 5. Dataset Overview

- Shape
- Column
- Head
- Tail
- Info

In [ ]:
print("Shape:", df.shape)

In [ ]:
print("Columns:")
print(df.columns.tolist())

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

## 6. EDA

`run_eda(df)` menjalankan keseluruhan tahap EDA sekaligus: dataset overview, missing value, duplikat, panjang review, distribusi rating & kategori, statistik dasar, serta pembuatan label sentimen (`sentiment` & `label`).

In [ ]:
eda_result = run_eda(df)
print_eda_summary(eda_result)

In [ ]:
df = eda_result["dataset_with_label"]
df[["text", "rating", "sentiment", "label"]].head()

## 7. EDA Visualization

In [ ]:
plot_class_distribution(df)

In [ ]:
plot_rating_count(df)

In [ ]:
plot_rating_pie(df)

In [ ]:
plot_review_length(df)

In [ ]:
plot_top_category(df)

## 8. Data Cleaning

`clean_text()` membersihkan teks: case folding, menghapus URL, HTML tag, email, mention, hashtag, angka, tanda baca, simbol, emoji, mojibake, dan karakter non-printable. Contoh penerapan pada satu sampel teks:

In [ ]:
sample_text = df["text"].iloc[0]
print("Sebelum :", sample_text)
print("Sesudah :", clean_text(sample_text))

## 9. Text Normalization

- `normalize_slang()` — mengubah kata slang/singkatan menjadi kata baku (`gk` → `tidak`, `yg` → `yang`, dst)
- `reduce_repeated_characters()` — mereduksi karakter berulang (`bagusss` → `bagus`)

In [ ]:
sample_clean = clean_text(sample_text)
sample_normalized_slang = normalize_slang(sample_clean)
sample_normalized = reduce_repeated_characters(sample_normalized_slang)

print("Setelah cleaning      :", sample_clean)
print("Setelah normalize_slang:", sample_normalized_slang)
print("Setelah reduce_repeated:", sample_normalized)

## 10. NLP Preprocessing

- `tokenize()` — memecah kalimat menjadi token/kata
- `remove_stopwords()` — menghapus kata umum yang tidak bermakna (stopword Bahasa Indonesia, gabungan NLTK + Sastrawi)
- `stemming()` — mengubah kata berimbuhan menjadi kata dasar (Sastrawi Stemmer)

In [ ]:
sample_tokens = tokenize(sample_normalized)
print("Tokens:", sample_tokens)

sample_tokens_no_stopword = remove_stopwords(sample_tokens)
print("\nSetelah remove_stopwords:", sample_tokens_no_stopword)

sample_stemmed = stemming(sample_tokens_no_stopword)
print("\nSetelah stemming:", sample_stemmed)

## 11. Apply Preprocessing

`apply_preprocessing(df)` menjalankan seluruh tahapan di atas (cleaning → normalisasi → tokenisasi → stopword removal → stemming) secara berurutan untuk semua baris pada dataset, menghasilkan kolom baru `clean_text`.

> **Catatan performa:** Proses stemming Sastrawi relatif lambat pada dataset besar. Untuk efisiensi waktu eksekusi, notebook ini menggunakan **subset (sample)** data secara default melalui variabel `SAMPLE_SIZE`. Untuk menjalankan pada seluruh dataset, set `SAMPLE_SIZE = None`.

In [ ]:
SAMPLE_SIZE = 8000  # ubah ke None untuk menggunakan seluruh dataset (lebih lama)

if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(df):
    df = (
        df.groupby("sentiment", group_keys=False)
        .apply(lambda x: x.sample(
            frac=SAMPLE_SIZE / len(df),
            random_state=RANDOM_STATE
        ))
        .reset_index(drop=True)
    )

print("Jumlah data yang diproses:", df.shape[0])
df["sentiment"].value_counts()

In [ ]:
df = apply_preprocessing(df)

df = df[df["clean_text"].str.strip() != ""].reset_index(drop=True)

print("Jumlah data setelah preprocessing:", df.shape[0])

## 12. Show Before After

In [ ]:
show_preprocessing_examples(df, n=10)

## 13. Train Test Split

Data dibagi 80:20 dengan **stratified split** berdasarkan label sentimen agar proporsi kelas pada data latih dan data uji tetap seimbang.

In [ ]:
X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

print("Data latih :", X_train.shape[0])
print("Data uji   :", X_test.shape[0])

LABELS = ["Negative", "Neutral", "Positive"]

---
## 14. Support Vector Machine (SVM)

`run_svm()` melatih dan mengevaluasi SVM dengan TF-IDF pada **4 kernel**: linear, RBF, polynomial, dan sigmoid, masing-masing dituning menggunakan `GridSearchCV`.

In [ ]:
svm_result = run_svm(
    X_train,
    X_test,
    y_train,
    y_test
)

### Kernel Comparison

In [ ]:
print_kernel_summary(svm_result)
plot_svm_kernel_comparison(svm_result["comparison_table"])

### Confusion Matrix

In [ ]:
best_svm = svm_result["best_result"]

plot_confusion_matrix(
    best_svm["confusion_matrix"],
    labels=LABELS,
    title=f"Confusion Matrix - SVM ({svm_result['best_kernel']} kernel)"
)

print("Classification Report:")
print(pd.DataFrame(best_svm["classification_report"]).transpose())

### Top TF-IDF Feature

In [ ]:
if svm_result["best_kernel"] == "Linear":
    feature_names = best_svm["model"].best_estimator_.named_steps["tfidf"].get_feature_names_out()
    plot_top_tfidf_features(best_svm["model"], feature_names)
else:
    print(f"Kernel terbaik adalah {svm_result['best_kernel']}, Top TF-IDF Feature hanya tersedia untuk kernel Linear.")

### Training Time

In [ ]:
plot_training_time(svm_result["comparison_table"])

---
## 15. CNN (Convolutional Neural Network)

`run_cnn()` membangun dan membandingkan dua arsitektur: **Conv1D** dan **Conv2D**.

Persiapan data: teks diubah ke sequence angka (`Tokenizer`), lalu **padding** ke panjang tetap. Untuk Conv2D, output `Embedding` di-**reshape** menjadi bentuk `(panjang, dimensi_embedding, 1)` agar sesuai input `Conv2D`.
`Conv2D` mengekstraksi fitur lokal lewat operasi konvolusi, `MaxPooling2D` melakukan downsampling (mengambil nilai maksimum tiap region), dan `Flatten` meratakan output multi-dimensi menjadi vektor 1D agar bisa diproses layer `Dense`.

In [ ]:
cnn_result = run_cnn(
    X_train,
    X_test,
    y_train,
    y_test
)

### Learning Curve

In [ ]:
best_cnn = cnn_result["best_result"]

plot_cnn_learning_curve(best_cnn["history"])

### Loss Curve

In [ ]:
plot_cnn_loss_curve(best_cnn["history"])

### Confusion Matrix

In [ ]:
plot_confusion_matrix(
    best_cnn["confusion_matrix"],
    labels=LABELS,
    title=f"Confusion Matrix - CNN ({cnn_result['best_architecture']})"
)

print("Classification Report:")
print(pd.DataFrame(best_cnn["classification_report"]).transpose())

---
## 16. LSTM (Long Short-Term Memory)

`run_lstm()` membangun model `Embedding → LSTM → Dropout → Dense → Dense(softmax)`.

Sama seperti CNN, teks diubah ke sequence angka dan di-**padding** ke panjang tetap. Tidak diperlukan reshape tambahan karena layer LSTM Keras menerima langsung input 3D `(batch, time_steps, features)` yang sudah dihasilkan `Embedding`.
LSTM memproses kata demi kata secara berurutan sambil menjaga "memori" (cell state) yang membawa konteks dari kata-kata sebelumnya, sehingga mampu menangkap ketergantungan jangka panjang antar kata dalam kalimat ulasan.

In [ ]:
lstm_result = run_lstm(
    X_train,
    X_test,
    y_train,
    y_test
)

### Learning Curve

In [ ]:
best_lstm = lstm_result["best_result"]

plot_lstm_learning_curve(best_lstm["history"])

### Loss Curve

In [ ]:
plot_lstm_loss_curve(best_lstm["history"])

### Confusion Matrix

In [ ]:
plot_confusion_matrix(
    best_lstm["confusion_matrix"],
    labels=LABELS,
    title="Confusion Matrix - LSTM"
)

print("Classification Report:")
print(pd.DataFrame(best_lstm["classification_report"]).transpose())

---
## 17. Final Comparison

Perbandingan Accuracy, Precision, Recall, F1-Score, dan Training Time dari ketiga model terbaik (SVM, CNN, LSTM).

In [ ]:
final_comparison = pd.DataFrame([
    {
        "Model": f"SVM ({svm_result['best_kernel']})",
        "Accuracy": best_svm["accuracy"],
        "Precision": best_svm["precision"],
        "Recall": best_svm["recall"],
        "F1-Score": best_svm["f1"],
        "Training Time (s)": best_svm["training_time"]
    },
    {
        "Model": f"CNN ({cnn_result['best_architecture']})",
        "Accuracy": best_cnn["accuracy"],
        "Precision": best_cnn["precision"],
        "Recall": best_cnn["recall"],
        "F1-Score": best_cnn["f1"],
        "Training Time (s)": best_cnn["training_time"]
    },
    {
        "Model": "LSTM",
        "Accuracy": best_lstm["accuracy"],
        "Precision": best_lstm["precision"],
        "Recall": best_lstm["recall"],
        "F1-Score": best_lstm["f1"],
        "Training Time (s)": best_lstm["training_time"]
    }
]).sort_values(by="F1-Score", ascending=False).reset_index(drop=True)

final_comparison

In [ ]:
plot_model_comparison(final_comparison)

---
## 18. WordCloud

Visualisasi kata-kata yang paling sering muncul pada masing-masing kelas sentimen.

In [ ]:
plot_wordcloud_by_sentiment(df)

## 19. Conclusion

- **SVM:** diuji pada 4 kernel (linear, RBF, polynomial, sigmoid) dengan representasi TF-IDF. Kernel terbaik dipilih berdasarkan F1-score tertinggi.
- **CNN:** dua arsitektur (Conv1D & Conv2D) dibangun dan dibandingkan. Conv2D/MaxPooling2D/Flatten bekerja mengekstraksi dan meringkas fitur lokal dari representasi embedding teks.
- **LSTM:** memanfaatkan kemampuan mengingat konteks sekuensial antar kata untuk klasifikasi sentimen.
- **Final Comparison** menunjukkan model dengan F1-score tertinggi sebagai model paling sesuai untuk kasus klasifikasi sentimen ulasan produk Tokopedia ini.
- Dataset cenderung **imbalanced** (didominasi rating/sentimen Positive), sehingga F1-score (weighted) dan classification report per kelas lebih representatif dibanding akurasi semata saat menginterpretasikan hasil.

> Catatan: hasil di atas menggunakan subset data (`SAMPLE_SIZE`) untuk efisiensi waktu eksekusi. Untuk hasil pada seluruh dataset, ubah `SAMPLE_SIZE = None` pada bagian 11, dengan konsekuensi waktu eksekusi (terutama GridSearchCV SVM dan training Sastrawi) yang jauh lebih lama.